# Laboratorio práctico · Semana 3 — Calidad de datos con Pandas
**Formación SENA | Caso integrador: seguimiento académico ficticio.**

Continuidad: Semana 1 (contexto, ética y ambiente) → Semana 2 (Series, DataFrames, CSV/Excel, inspección) → Semana 3 (perfilamiento, decisiones de calidad y trazabilidad).

**Importante:** los datos son inventados y las reglas de oportunidad/validez son académicas, no reglamentos institucionales.

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"
PROCESSED = RAIZ / "data" / "processed"
OUTPUTS = RAIZ / "outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
archivo = RAW / "seguimiento_academico_semana3_con_errores.csv"
assert archivo.exists(), f"Ubique el CSV de la actividad en {archivo}"
print("Versión Pandas:", pd.__version__)
print("Fuente:", archivo.name)

Versión Pandas: 3.0.6
Fuente: seguimiento_academico_semana3_con_errores.csv


## 1. Cargar y perfilar la fuente (sin modificar el original)

In [2]:
df_raw = pd.read_csv(archivo, dtype={"Documento": "string", "Ficha": "string"})
df_raw.insert(0, "fila_fuente", range(2, len(df_raw) + 2))
columnas_fuente = [col for col in df_raw.columns if col != "fila_fuente"]
print("Dimensiones:", df_raw.shape)
print(df_raw.head(4).to_string(index=False))
df_raw.info()
print(df_raw.describe(include="all").to_string())

Dimensiones: (20, 10)
 fila_fuente Documento Nombre Programa  Ficha Nota Python  Nota Datos  Asistencia (%)   Ciudad Fecha Actualizacion
           2      1001    Ana     ADSO 287001         4.2         4.0            92.0 Medellín          2026-10-12
           3      1002 Carlos     ADSO 287001         3.8         3.6            85.0    Bello          2026-10-11
           4      1003  María     ADSO 287001         4.7         4.8            96.0   Itagüí          2026-10-12
           5      1004   Luis     ADSO 287001         2.9         3.1            70.0 Medellín          2026-10-10
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   fila_fuente          20 non-null     int64  
 1   Documento            20 non-null     string 
 2   Nombre               20 non-null     str    
 3   Programa             20 non-null     str    
 4   Ficha 

## 2. Valores nulos y duplicados completos versus claves repetidas

In [3]:
print("Faltantes por columna:")
print(df_raw.isna().sum())
print("isnull() produce el mismo resultado:",
      df_raw.isna().equals(df_raw.isnull()))
exactos = df_raw.duplicated(subset=columnas_fuente, keep="first")
print("Duplicados exactos sobrantes:", int(exactos.sum()))
print(df_raw.loc[exactos, ["fila_fuente", "Documento", "Nombre"]])

Faltantes por columna:
fila_fuente            0
Documento              0
Nombre                 0
Programa               0
Ficha                  0
Nota Python            1
Nota Datos             1
Asistencia (%)         1
Ciudad                 1
Fecha Actualizacion    0
dtype: int64
isnull() produce el mismo resultado: True
Duplicados exactos sobrantes: 1
    fila_fuente Documento  Nombre
18           20      1002  Carlos


**Decisión:** retirar solamente las filas exactamente repetidas y conservar su reporte.
No eliminar automáticamente claves de negocio repetidas si los datos son diferentes.

In [4]:
reporte_exactos = df_raw.loc[exactos].copy()
df = df_raw.loc[~exactos].copy()
df = df.rename(columns={
    "Documento": "documento", "Nombre": "nombre", "Programa": "programa",
    "Ficha": "ficha", "Nota Python": "nota_python",
    "Nota Datos": "nota_datos", "Asistencia (%)": "asistencia",
    "Ciudad": "ciudad", "Fecha Actualizacion": "fecha_actualizacion"
})
for col in ["documento", "nombre", "programa", "ficha", "ciudad", "fecha_actualizacion"]:
    df[col] = df[col].astype("string").str.strip().replace("", pd.NA)
claves_repetidas = df.duplicated(subset=["documento"], keep=False)
print("Filas con documento repetido (pendientes de revisión):", int(claves_repetidas.sum()))
print(df.loc[claves_repetidas, ["fila_fuente", "documento", "nombre", "nota_python"]].to_string(index=False))

Filas con documento repetido (pendientes de revisión): 2
 fila_fuente documento nombre nota_python
           8      1007 Camila         4,0
          21      1007 Camila         4.2


## 3. Conversión segura de tipos y normalización de categorías

In [5]:
for col in ["nota_python", "nota_datos"]:
    s = df[col].astype("string").str.strip().str.replace(",", ".", regex=False)
    df[col] = pd.to_numeric(s, errors="coerce").astype("Float64")
s = df["asistencia"].astype("string").str.strip().str.replace("%", "", regex=False)
df["asistencia"] = pd.to_numeric(s, errors="coerce").astype("Float64")
df["fecha_actualizacion"] = pd.to_datetime(
    df["fecha_actualizacion"], format="%Y-%m-%d", errors="coerce"
)
df["documento"] = df["documento"].astype("string")
df["ficha"] = df["ficha"].astype("string")
print(df[["documento", "nota_python", "asistencia", "fecha_actualizacion"]].dtypes)

documento                      string
nota_python                   Float64
asistencia                    Float64
fecha_actualizacion    datetime64[us]
dtype: object


In [6]:
mapa_programa = {
    "adso": "ADSO",
    "ciencia datos python": "Ciencia de datos con Python",
    "ciencia de datos con python": "Ciencia de datos con Python"
}
mapa_ciudad = {
    "medellín": "Medellín", "medellin": "Medellín",
    "bello": "Bello", "itagüí": "Itagüí", "itagui": "Itagüí",
    "envigado": "Envigado"
}
df["programa_normalizado"] = df["programa"].str.casefold().map(mapa_programa)
df["ciudad_normalizada"] = df["ciudad"].str.casefold().map(mapa_ciudad)
print(df[["programa", "programa_normalizado", "ciudad", "ciudad_normalizada"]].tail(10).to_string(index=False))

                   programa        programa_normalizado   ciudad ciudad_normalizada
                       ADSO                        ADSO    Bello              Bello
                       ADSO                        ADSO medellín           Medellín
                       ADSO                        ADSO    BELL0                NaN
       Ciencia Datos Python Ciencia de datos con Python     <NA>                NaN
                       ADSO                        ADSO Envigado           Envigado
Ciencia de datos con Python Ciencia de datos con Python   Itagui             Itagüí
                       ADSO                        ADSO Medellín           Medellín
                       ADSO                        ADSO    Bello              Bello
                       ADSO                        ADSO medellin           Medellín
                       ADSO                        ADSO Medellín           Medellín


No convierta automáticamente `BELL0` en `Bello`: la similitud no confirma que sea el mismo valor. Registre la anomalía. Mantenga también la columna original.

## 4. Reglas: completitud, unicidad, consistencia, validez, oportunidad y trazabilidad

In [7]:
fecha_corte = pd.Timestamp("2026-10-18")  # Fecha didáctica fija para reproducibilidad
faltantes_clave = df[["documento", "programa_normalizado"]].isna().any(axis=1)
nota_python_fuera = df["nota_python"].notna() & ~df["nota_python"].between(0, 5)
nota_datos_fuera = df["nota_datos"].notna() & ~df["nota_datos"].between(0, 5)
asistencia_fuera = df["asistencia"].notna() & ~df["asistencia"].between(0, 100)
ciudad_no_reconocida = df["ciudad"].notna() & df["ciudad_normalizada"].isna()
fecha_no_vigente = ~df["fecha_actualizacion"].between(
    fecha_corte - pd.Timedelta(days=30), fecha_corte
)
incidencias = pd.DataFrame({
    "fila_fuente": df["fila_fuente"], "documento": df["documento"],
    "falta_clave": faltantes_clave, "documento_repetido": claves_repetidas,
    "nota_python_fuera_rango": nota_python_fuera,
    "nota_datos_fuera_rango": nota_datos_fuera,
    "asistencia_fuera_rango": asistencia_fuera,
    "ciudad_no_reconocida": ciudad_no_reconocida,
    "fecha_fuera_ventana": fecha_no_vigente
})
print(incidencias.select_dtypes(include="bool").sum().to_string())

falta_clave                0
documento_repetido         2
nota_python_fuera_rango    2
nota_datos_fuera_rango     0
asistencia_fuera_rango     2
ciudad_no_reconocida       1
fecha_fuera_ventana        3


**Completitud:** presencia de campos necesarios. **Unicidad:** una clave por entidad si la regla lo exige. **Consistencia:** categorías/relaciones coherentes. **Validez:** dominio/rangos. **Oportunidad:** vigencia con fecha de corte. **Trazabilidad:** fuente, filas y transformaciones.
Las reglas son didácticas y deben revisarse con la necesidad de negocio.

## 5. Tratamiento justificado: dropna(), fillna() y astype()

In [8]:
df_preparado = df.copy()
# No inventar notas: registrar valores fuera de rango como ausentes.
for col, mascara in [
    ("nota_python", nota_python_fuera),
    ("nota_datos", nota_datos_fuera),
    ("asistencia", asistencia_fuera)
]:
    df_preparado.loc[mascara, col] = pd.NA
# Demostrar fillna() en una columna de presentación; no reemplazar la fuente.
df_preparado["ciudad_reporte"] = df_preparado["ciudad_normalizada"].fillna("No reportada")
# Demostrar dropna() solo para un subconjunto que exige claves completas.
df_con_claves = df_preparado.dropna(subset=["documento", "programa_normalizado"])
# Tipos enteros anulables; no convertir documentos a número.
df_preparado["asistencia"] = df_preparado["asistencia"].astype("Int64")
df_preparado["archivo_fuente"] = archivo.name
df_preparado["fecha_corte"] = fecha_corte.strftime("%Y-%m-%d")
print("Filas preparadas:", len(df_preparado))
print("Filas para análisis con claves completas:", len(df_con_claves))

Filas preparadas: 19
Filas para análisis con claves completas: 19


**Atención:** `df_preparado` NO es una base certificada: conserva registros con claves contradictorias y anomalías documentadas. Estos casos requieren verificación de la fuente antes de decidir si se consolidan, excluyen o corrigen.

## 6. Exportar evidencias y validar el resultado

In [9]:
salida_preparada = PROCESSED / "seguimiento_academico_semana3_preparado.csv"
salida_incidencias = OUTPUTS / "reporte_incidencias_semana3.csv"
salida_duplicados = OUTPUTS / "duplicados_exactos_retirados.csv"
df_preparado.to_csv(salida_preparada, index=False, encoding="utf-8-sig")
incidencias.to_csv(salida_incidencias, index=False, encoding="utf-8-sig")
reporte_exactos.to_csv(salida_duplicados, index=False, encoding="utf-8-sig")
# Verificación de calidad de la exportación.
verificacion = pd.read_csv(salida_preparada, dtype={"documento": "string", "ficha": "string"})
assert len(verificacion) == len(df_preparado)
assert verificacion["documento"].astype("string").equals(df_preparado["documento"].reset_index(drop=True))
print("Exportaciones correctas:", salida_preparada.name, salida_incidencias.name)

Exportaciones correctas: seguimiento_academico_semana3_preparado.csv reporte_incidencias_semana3.csv


## Reflexión y sustentación
1. ¿Cuántas filas se recibieron y cuántas se prepararon? Explique la diferencia.
2. ¿Qué dato no debe imputarse sin evidencia y por qué?
3. ¿Por qué un documento repetido con notas distintas no se elimina automáticamente?
4. ¿Qué mide la oportunidad y cómo cambia al cambiar la fecha de corte?
5. ¿Qué aspectos quedarían pendientes antes de conectar estos datos a un sistema real?

## 📌 Reflexión y Sustentación Técnica (Semana 3)

1. **¿Cuántas filas se recibieron y cuántas se prepararon? Explique la diferencia.**
   - Se recibieron las filas del archivo original (`df_raw`).
   - Se prepararon únicamente las filas tras retirar las **duplicadas exactas** (`reporte_exactos`), conservando los registros únicos en `df_preparado`.

2. **¿Qué dato no debe imputarse sin evidencia y por qué?**
   - Las **notas académicas (`nota_python`, `nota_datos`)** y los **documentos de identidad**. Imputarle una nota a un aprendiz inventaría un rendimiento académico irreal, afectando la validez del reporte.

3. **¿Por qué un documento repetido con notas distintas no se elimina automáticamente?**
   - Porque puede tratarse de un error de digitación en la clave o un problema de actualización. Eliminarlo a ciegas provocaría la pérdida irrecuperable de información válida de un aprendiz.

4. **¿Qué mide la oportunidad y cómo cambia al cambiar la fecha de corte?**
   - Mida la **vigencia temporal** del dato. Si se cambia la `fecha_corte` (ej. 2026-10-18), cambia la ventana de 30 días, lo que hace que registros antiguos pasen a ser considerados fuera de norma o no vigentes.

5. **¿Qué aspectos quedarían pendientes antes de conectar estos datos a un sistema real?**
   - Definir reglas para resolver conflictos de claves duplicadas, validar con la fuente los valores fuera de rango y automatizar un pipeline de validación para nuevos archivos.